# Benchmarking Pipeline with Unlsoth

## Initial Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
!pip install -q torch transformers pandas tqdm bitsandbytes accelerate unsloth  # Takes about 7-10 mins depending on internet connection

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.0/353.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 803.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os
import re
import gc
import json
import time
import torch

import pandas as pd
from tqdm import tqdm
from pathlib import Path
from unsloth import FastLanguageModel
from transformers import pipeline, infer_device, AutoTokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# LLM Model Benchmarking Pipeline

In [43]:
class Benchmark:
    def __init__(self, model_id, input_data, batch_size):
        self.model_name = self.get_model_name(model_id)
        self.model, self.tokenizer = self.load_model()
        self.batch_size = batch_size
        self.input_file = input_data

    def get_model_name(self, model_id: str) -> str:
        model_dict = {
            "llama": "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
            "gemma": "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
            "mistral": "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
        }
        if model_id.strip().lower() not in model_dict.keys():
            raise ValueError(f"Invalid Model Name, expected {list(model_dict.keys())} got {model_id}")

        return model_dict[model_id]

    def load_model(self):
        """Loads model_id"""
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

        max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
        dtype = None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
        load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = self.model_name,
            max_seq_length = max_seq_length,
            dtype = dtype,
            load_in_4bit = load_in_4bit,
        )

        tokenizer.padding_side = "left"
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.truncation_side = "left"

        return model, tokenizer

    def json_to_dict(self, file_path: str) -> dict:
        """Converts JSON file to dictionary"""
        data_dict = {}
        with open(file_path, 'r') as data:
            data_dict = json.load(data)

        return data_dict

    def csv_to_dict(self, file_path: str) -> dict:
        """Converts CSV file to dictionary"""
        data_list = []
        data_dict = {}

        with open(file_path, 'r') as data:
            data_list = data.readlines()

        for elements in data_list[1:]:
            id, sys, prompt = elements.split(',')
            data_dict[id] = {
                "system_prompt": sys.strip(),
                "test_prompt": prompt.strip()
            }

        return data_dict

    def extract_responses(self, outputs: list) -> list:
        """
        Extract only the text generated in the ### Response: section from each output.
        """
        responses = []

        for out in outputs:
            text = out

            # Remove all special tokens like <|begin_of_text|>, <|finetune_right_pad_id|>
            text = re.sub(r"<\|.*?\|>", "", text)

            text = re.sub(r"</s>|<s>", "", text)

            # Extract everything after the last ### Response:
            if "### Response:" in text:
                # Take text after the last occurrence of ### Response:
                text = text.split("### Response:")[-1]

            # Remove everything after the next ### Input (if any)
            text = text.split("### Input:")[0]

            # Strip leading/trailing whitespace
            text = text.strip()

            responses.append(text)

        return responses

    def load_data(self) -> dict:
        """
        Returns a dictionary of the data.
        """
        path = Path(self.input_file)
        filename, filetype = path.name.split('.')  # abc.xyz
        data_dir = path.parent.parent  # Parent of input dir [CSDS555/data/]

        out_name = f"{filename}_{time.time_ns()}.json"  # output filename same as input with timestamp
        output_dir = os.path.join(data_dir, "output")  # Output dir [CSDS555/data/output/]
        output_file = os.path.join(output_dir, out_name)

        # Create output dir if it doesn't exist
        if not os.path.exists(output_dir):
            print(f"Creating output directory: {output_dir}")
            os.makedirs(output_dir)

        # TODO:
        # Instead of above switch to appropriate data_loader
        # Make Sure the content follows similar format as
        # unique_id,system_prompt,prompt is CSV
        # OR
        # "01": {
        #     "system_prompt": "Always answer max in 2 lines.",
        #     "test_prompt": "You are an expert mathematician,solve this calculus problem: find the derivative of x^3 + 4x^2 - 7x + 10"
        # }, if JSON

        # Step 1: Load data
        data_dict = {}
        if filetype == "csv":
            data_dict = self.csv_to_dict(self.input_file)
        elif filetype == "json":
            data_dict = self.json_to_dict(self.input_file)
        else:
            raise TypeError(f"Expected filetype 'csv' or 'json', got {filetype}")

        return data_dict

    def load_prompt(self) -> list:
        """
        Returns a list of batches of data.
        """
        alpaca_prompt = """
        ### Instruction: {}

        ### Input: {}

        ### Response:
        {}
        """
        return alpaca_prompt

    def run(self):
        """
        Runs the benchmark
        """
        # TODO (for enlosed): Load data in list of dict as batches, use self.batch_size
        # TODO (for enlosed): This batching can be made better and mem efficient based on data loading style
        # ______________________________________________________________________________________
        # Loads the data
        data_dict = self.load_data()

        # Converting ids to batches
        ids = list(data_dict.keys())
        batches = [ids[i:i+self.batch_size] for i in range(0, len(ids), self.batch_size)]
        # ______________________________________________________________________________________

        # Load the model for inferencing
        FastLanguageModel.for_inference(self.model)  # Enable native 2x faster inference

        # Format the prompt in batch
        prompt_format = self.load_prompt()
        eos_token = self.tokenizer.eos_token
        all_outputs = []

        for batch_ids in tqdm(batches, desc="Processing Batches"):
            # Create prompts for inputs for batch
            prompts = [
                prompt_format.format(
                    data_dict[i]["system_prompt"],
                    data_dict[i]["test_prompt"],
                    ""
                )
                for i in batch_ids
            ]

            inputs = self.tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to("cuda")

            # Inference on batch
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=64,
                use_cache=True
            )

            decoded = self.tokenizer.batch_decode(outputs)

            # Post process batch
            processed_batch = self.extract_responses(decoded)

            # Store aligned with original IDs
            for i, out in zip(batch_ids, processed_batch):
                all_outputs.append((i, out))  # TODO: Instead of list append store to DB or something

        return all_outputs

## Execution Cell

In [44]:
MODEL_NAME = "mistral"  # Use llama, mistral or gemma
input_path = "/content/drive/MyDrive/WPI/CS_555/Project/data/input/test_data.json"
BATCH_SIZE = 8

In [45]:
runner = Benchmark(MODEL_NAME, input_path, BATCH_SIZE)  # Model downloading takes 10-ish minutes for first time
results = runner.run()

==((====))==  Unsloth 2025.11.3: Fast Mistral patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Processing Batches: 100%|██████████| 3/3 [00:37<00:00, 12.55s/it]


In [46]:
results[:5]

[('01',
  '1. First, simplify the expression: 3x^2 + 8x - 7\n        2. To find the derivative, apply the power rule: d/dx (3x^2) = 6x, d/dx (8x) = 8, and d/dx'),
 ('02',
  "1. Newton's third law states that for every action, there is an equal and opposite reaction.\n        2. In the context of rocket propulsion, when a rocket expels exhaust gases, the exhaust gases exert a forward force on the rocket (action).\n        3."),
 ('03',
  '1. Identify the oxidation and reduction reactions in the given redox reaction.\n\n        2. Balance the number of atoms in each element in both the oxidation and reduction reactions separately. Then, balance the number of electrons in the overall reaction.\n\n        3. Write the balanced'),
 ('04',
  '1. Photosynthesis is a process carried out by green plants, algae, and some bacteria. It uses light energy, usually from the sun, to convert carbon dioxide and water into glucose and oxygen.\n\n        2. This process occurs in two main stages: the ligh

In [47]:
len(results)

20